## 1. Environment Setup & Imports

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import degree, to_undirected, random_walk

# Project root (Kaggle)
PROJECT_ROOT = Path("/kaggle/working/hybrid_multimodal_retrieval").resolve()

# Add to path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph.config import load_entity_graph_config, get_entity_graph_config
from src.graph.build_entity_graph import load_entity_graph

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load Configuration & Artifacts

In [ ]:
# Load configuration
config_path = PROJECT_ROOT / "configs" / "entity_graph.yaml"
print(f"\nLoading config from: {config_path}")
cfg = load_entity_graph_config(str(config_path))
cfg_entity = get_entity_graph_config(cfg)

print(f"  k_sem: {cfg_entity['k_sem']}")
print(f"  degree_cap: {cfg_entity['degree_cap']}")

# Resolve artifact paths (using Kaggle input path for data files)
KAGGLE_INPUT = Path("/kaggle/input/flickr30k")
entity_meta_path = KAGGLE_INPUT / cfg_entity["entity_meta_path"]
entity_context_path = KAGGLE_INPUT / cfg_entity["context_path"]
graph_path = KAGGLE_INPUT / cfg_entity["entity_graph_path"]

print(f"\nArtifact paths:")
print(f"  Metadata: {entity_meta_path}")
print(f"  Context: {entity_context_path}")
print(f"  Graph: {graph_path}")

In [ ]:
# Load entity metadata
print("\nLoading entity_meta.json...")
with entity_meta_path.open("r", encoding="utf-8") as f:
    entity_meta_raw = json.load(f)

# Convert string keys to int
entity_meta = {int(eid): meta for eid, meta in entity_meta_raw.items()}
print(f"  ✓ Loaded metadata for {len(entity_meta)} entities")

# Load entity context
print("\nLoading entity_context.json...")
with entity_context_path.open("r", encoding="utf-8") as f:
    entity_context_raw = json.load(f)

# Convert string keys to int
entity_context = {int(eid): ctx for eid, ctx in entity_context_raw.items()}
print(f"  ✓ Loaded context for {len(entity_context)} entities")

# Load entity graph
print("\nLoading entity_graph.pt...")
graph = load_entity_graph(graph_path)
print(f"  ✓ Loaded graph")

In [ ]:
# Print graph summary
print("\n" + "=" * 70)
print("ENTITY GRAPH SUMMARY")
print("=" * 70)

# Node features
num_nodes = graph["entity"].x.size(0)
feat_dim = graph["entity"].x.size(1)
feat_dtype = graph["entity"].x.dtype

print(f"\nNodes:")
print(f"  Type: 'entity'")
print(f"  Count: {num_nodes:,}")
print(f"  Feature dim: {feat_dim}")
print(f"  Feature dtype: {feat_dtype}")

# Semantic edges
sem_ei = graph["entity", "sem", "entity"].edge_index
sem_ew = graph["entity", "sem", "entity"].edge_weight
num_sem_edges = sem_ei.size(1)

print(f"\nSemantic Edges ('entity', 'sem', 'entity'):")
print(f"  Count: {num_sem_edges:,}")
print(f"  Edge index shape: {list(sem_ei.shape)}")
print(f"  Edge weight shape: {list(sem_ew.shape)}")
print(f"  Edge weight dtype: {sem_ew.dtype}")

# Co-occurrence edges
cooc_ei = graph["entity", "cooc", "entity"].edge_index
cooc_ew = graph["entity", "cooc", "entity"].edge_weight
num_cooc_edges = cooc_ei.size(1)

print(f"\nCo-occurrence Edges ('entity', 'cooc', 'entity'):")
print(f"  Count: {num_cooc_edges:,}")
print(f"  Edge index shape: {list(cooc_ei.shape)}")
print(f"  Edge weight shape: {list(cooc_ew.shape)}")
print(f"  Edge weight dtype: {cooc_ew.dtype}")

print("=" * 70)

## 3. ID Alignment & Metadata/Context Sanity Checks

In [ ]:
print("\n" + "=" * 70)
print("ID ALIGNMENT CHECKS")
print("=" * 70)

# Compute ID sets
all_ids = set(range(num_nodes))
meta_ids = set(entity_meta.keys())
ctx_ids = set(entity_context.keys())

print(f"\nID counts:")
print(f"  Graph nodes: {len(all_ids):,}")
print(f"  Metadata entries: {len(meta_ids):,}")
print(f"  Context entries: {len(ctx_ids):,}")

# Check for missing IDs
missing_meta = all_ids - meta_ids
missing_ctx = all_ids - ctx_ids

print(f"\nMissing from metadata: {len(missing_meta)}")
print(f"Missing from context: {len(missing_ctx)}")

if missing_meta:
    print(f"  ✗ ERROR: {len(missing_meta)} node IDs missing from metadata!")
    print(f"    First 10: {sorted(missing_meta)[:10]}")
    raise AssertionError("Node IDs missing from metadata")

if missing_ctx:
    print(f"  ✗ ERROR: {len(missing_ctx)} node IDs missing from context!")
    print(f"    First 10: {sorted(missing_ctx)[:10]}")
    raise AssertionError("Node IDs missing from context")

# Check for extra IDs
extra_meta = meta_ids - all_ids
extra_ctx = ctx_ids - all_ids

print(f"\nExtra in metadata: {len(extra_meta)}")
print(f"Extra in context: {len(extra_ctx)}")

if extra_meta:
    print(f"  ⚠ Warning: {len(extra_meta)} IDs in metadata not present in graph")
    print(f"    First 10: {sorted(extra_meta)[:10]}")

if extra_ctx:
    print(f"  ⚠ Warning: {len(extra_ctx)} IDs in context not present in graph")
    print(f"    First 10: {sorted(extra_ctx)[:10]}")

if not missing_meta and not missing_ctx:
    print("\n  ✓ All node IDs are present in both metadata and context!")

print("=" * 70)

## 4. Degree Distributions (Semantic & Co-occurrence)

In [ ]:
print("\n" + "=" * 70)
print("DEGREE DISTRIBUTIONS")
print("=" * 70)

# Semantic degree statistics
print("\n[Semantic Edges]")

sem_deg_out = degree(sem_ei[0], num_nodes=num_nodes)
sem_deg_in = degree(sem_ei[1], num_nodes=num_nodes)
sem_ei_undirected = to_undirected(sem_ei)
sem_deg_undirected = degree(sem_ei_undirected[0], num_nodes=num_nodes)

print(f"  Out-degree:  mean={sem_deg_out.mean():.2f}, min={sem_deg_out.min():.0f}, max={sem_deg_out.max():.0f}")
print(f"  In-degree:   mean={sem_deg_in.mean():.2f}, min={sem_deg_in.min():.0f}, max={sem_deg_in.max():.0f}")
print(f"  Undirected:  mean={sem_deg_undirected.mean():.2f}, min={sem_deg_undirected.min():.0f}, max={sem_deg_undirected.max():.0f}")

sem_isolated = (sem_deg_undirected == 0).sum().item()
print(f"  Isolated nodes: {sem_isolated:,} ({100.0 * sem_isolated / num_nodes:.2f}%)")

# Co-occurrence degree statistics
print("\n[Co-occurrence Edges]")

if num_cooc_edges > 0:
    cooc_deg_out = degree(cooc_ei[0], num_nodes=num_nodes)
    cooc_deg_in = degree(cooc_ei[1], num_nodes=num_nodes)
    cooc_ei_undirected = to_undirected(cooc_ei)
    cooc_deg_undirected = degree(cooc_ei_undirected[0], num_nodes=num_nodes)
    
    print(f"  Out-degree:  mean={cooc_deg_out.mean():.2f}, min={cooc_deg_out.min():.0f}, max={cooc_deg_out.max():.0f}")
    print(f"  In-degree:   mean={cooc_deg_in.mean():.2f}, min={cooc_deg_in.min():.0f}, max={cooc_deg_in.max():.0f}")
    print(f"  Undirected:  mean={cooc_deg_undirected.mean():.2f}, min={cooc_deg_undirected.min():.0f}, max={cooc_deg_undirected.max():.0f}")
    
    cooc_isolated = (cooc_deg_undirected == 0).sum().item()
    print(f"  Isolated nodes: {cooc_isolated:,} ({100.0 * cooc_isolated / num_nodes:.2f}%)")
else:
    print("  No co-occurrence edges found!")

print("=" * 70)

In [ ]:
# Plot semantic degree distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(sem_deg_undirected.cpu().numpy(), bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Degree')
plt.ylabel('Frequency')
plt.title('Semantic Edge Degree Distribution (Undirected)')
plt.grid(axis='y', alpha=0.3)

# Plot co-occurrence degree distribution
plt.subplot(1, 2, 2)
if num_cooc_edges > 0:
    plt.hist(cooc_deg_undirected.cpu().numpy(), bins=50, edgecolor='black', alpha=0.7, color='orange')
    plt.xlabel('Degree')
    plt.ylabel('Frequency')
    plt.title('Co-occurrence Edge Degree Distribution (Undirected)')
    plt.grid(axis='y', alpha=0.3)
else:
    plt.text(0.5, 0.5, 'No co-occurrence edges', ha='center', va='center', fontsize=14)
    plt.title('Co-occurrence Edge Degree Distribution')
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

## 5. Connectivity Analysis (Semantic Graph)

In [ ]:
print("\n" + "=" * 70)
print("CONNECTIVITY ANALYSIS (Semantic Graph)")
print("=" * 70)

# Build undirected NetworkX graph from semantic edges
print("\nBuilding NetworkX graph from semantic edges...")
G_sem = nx.Graph()
G_sem.add_nodes_from(range(num_nodes))
G_sem.add_edges_from(zip(sem_ei_undirected[0].tolist(), sem_ei_undirected[1].tolist()))
print(f"  ✓ Graph built: {G_sem.number_of_nodes():,} nodes, {G_sem.number_of_edges():,} edges")

# Compute connected components
print("\nComputing connected components...")
num_components = nx.number_connected_components(G_sem)
components = sorted(nx.connected_components(G_sem), key=len, reverse=True)

print(f"  Number of components: {num_components:,}")

# Largest component
largest_comp = components[0]
largest_size = len(largest_comp)
largest_pct = 100.0 * largest_size / num_nodes

print(f"\n  Largest component:")
print(f"    Size: {largest_size:,} nodes ({largest_pct:.2f}% of total)")

# Top 10 components
print(f"\n  Top 10 component sizes:")
for i, comp in enumerate(components[:10], 1):
    size = len(comp)
    pct = 100.0 * size / num_nodes
    print(f"    {i:2d}. {size:6,} nodes ({pct:5.2f}%)")

# Small components
small_comps = [c for c in components if len(c) < 10]
if small_comps:
    print(f"\n  Small components (< 10 nodes): {len(small_comps)}")
    singleton_comps = [c for c in components if len(c) == 1]
    print(f"    Singletons: {len(singleton_comps)}")

print("=" * 70)

## 6. Entity-Level Inspection (Metadata + Context)

In [ ]:
def summarize_entity(eid: int):
    """Print summary of entity metadata and context."""
    if eid not in entity_meta or eid not in entity_context:
        print(f"  ✗ Entity {eid} not found in metadata or context")
        return
    
    meta = entity_meta[eid]
    ctx = entity_context[eid]
    
    print(f"\nEntity {eid}:")
    print(f"  Name: {meta['name']!r}")
    print(f"  Collection frequency: {meta['cf']}")
    print(f"  Document frequency (images): {meta['df_image']}")
    print(f"  Document frequency (captions): {meta['df_caption']}")
    print(f"  Image IDs: {len(ctx.get('image_ids', []))} images")
    print(f"  Caption IDs: {len(ctx.get('caption_ids', []))} captions")

In [ ]:
print("\n" + "=" * 70)
print("ENTITY METADATA & CONTEXT SAMPLES")
print("=" * 70)

# Show sample entities
sample_ids = [0, 1, 2, 42, 100, 500, 1000]
sample_ids = [eid for eid in sample_ids if eid < num_nodes]

for eid in sample_ids:
    summarize_entity(eid)

print("\n" + "=" * 70)

## 7. Neighbor Inspection (Semantic & Co-occurrence)

In [ ]:
def inspect_neighbors(eid: int, top_k: int = 10):
    """
    Inspect semantic and co-occurrence neighbors of an entity.
    
    Args:
        eid: Entity ID to inspect
        top_k: Number of top neighbors to show
    """
    if eid not in entity_meta:
        print(f"  ✗ Entity {eid} not found")
        return
    
    name = entity_meta[eid]["name"]
    print(f"\n{'=' * 70}")
    print(f"NEIGHBORS OF ENTITY {eid}: {name!r}")
    print("=" * 70)
    
    # Semantic neighbors
    print(f"\n[Semantic Neighbors (top {top_k})]")
    sem_mask = (sem_ei[0] == eid)
    sem_neighbor_ids = sem_ei[1, sem_mask]
    sem_weights = sem_ew[sem_mask]
    
    if len(sem_neighbor_ids) > 0:
        # Sort by weight descending
        sorted_indices = torch.argsort(sem_weights, descending=True)
        top_indices = sorted_indices[:top_k]
        
        for i, idx in enumerate(top_indices, 1):
            nid = sem_neighbor_ids[idx].item()
            weight = sem_weights[idx].item()
            nname = entity_meta[nid]["name"]
            print(f"  {i:2d}. Entity {nid:5d} | weight={weight:.4f} | {nname!r}")
    else:
        print("  (no semantic neighbors)")
    
    # Co-occurrence neighbors
    print(f"\n[Co-occurrence Neighbors (top {top_k})]")
    if num_cooc_edges > 0:
        cooc_mask = (cooc_ei[0] == eid)
        cooc_neighbor_ids = cooc_ei[1, cooc_mask]
        cooc_weights = cooc_ew[cooc_mask]
        
        if len(cooc_neighbor_ids) > 0:
            # Sort by weight descending
            sorted_indices = torch.argsort(cooc_weights, descending=True)
            top_indices = sorted_indices[:top_k]
            
            for i, idx in enumerate(top_indices, 1):
                nid = cooc_neighbor_ids[idx].item()
                weight = cooc_weights[idx].item()
                nname = entity_meta[nid]["name"]
                print(f"  {i:2d}. Entity {nid:5d} | weight={weight:.4f} | {nname!r}")
        else:
            print("  (no co-occurrence neighbors)")
    else:
        print("  (no co-occurrence edges in graph)")

In [ ]:
# Inspect neighbors for sample entities
sample_ids = [0, 1, 42, 100, 500]
sample_ids = [eid for eid in sample_ids if eid < num_nodes]

for eid in sample_ids:
    inspect_neighbors(eid, top_k=5)

## 8. Random Walks on Semantic Graph

In [ ]:
def run_random_walk(start_eid: int, walk_length: int = 8, num_walks: int = 3):
    """
    Run random walks starting from a given entity.
    
    Args:
        start_eid: Starting entity ID
        walk_length: Length of each walk
        num_walks: Number of walks to perform
    """
    if start_eid not in entity_meta:
        print(f"  ✗ Entity {start_eid} not found")
        return
    
    name = entity_meta[start_eid]["name"]
    print(f"\n{'=' * 70}")
    print(f"RANDOM WALKS FROM ENTITY {start_eid}: {name!r}")
    print(f"  Walk length: {walk_length}, Number of walks: {num_walks}")
    print("=" * 70)
    
    # Create start nodes tensor
    start_nodes = torch.full((num_walks,), start_eid, dtype=torch.long)
    
    # Perform random walks
    walks = random_walk(
        sem_ei_undirected[0],
        sem_ei_undirected[1],
        start_nodes,
        walk_length=walk_length
    )
    
    # Display walks
    for i, path in enumerate(walks, 1):
        ids = path.tolist()
        names = [entity_meta[int(eid)]["name"] if int(eid) >= 0 else "N/A" for eid in ids]
        
        print(f"\nWalk {i}:")
        print(f"  IDs:   {ids}")
        print(f"  Names: {names}")

In [ ]:
# Run random walks from sample entities
sample_ids = [0, 1, 42, 100]
sample_ids = [eid for eid in sample_ids if eid < num_nodes]

for eid in sample_ids:
    run_random_walk(eid, walk_length=6, num_walks=2)

## 8. Text-Based Entity Search & Rich Examples

### 8.1 Helper: Find Entities by Name Substring

In [ ]:
def find_entities(substr: str, limit: int = 20, case_sensitive: bool = False):
    """
    Find entities whose name contains the given substring.
    
    Args:
        substr: Search substring, e.g. "grass", "dog", "man".
        limit: Max number of matches to print.
        case_sensitive: If False (default), match case-insensitively.
    """
    if not substr:
        print("Please provide a non-empty search string.")
        return
    
    if not case_sensitive:
        key = substr.lower()
        matches = [
            (eid, meta["name"])
            for eid, meta in entity_meta.items()
            if key in meta["name"].lower()
        ]
    else:
        matches = [
            (eid, meta["name"])
            for eid, meta in entity_meta.items()
            if substr in meta["name"]
        ]
    
    print(f"Found {len(matches)} matches for {substr!r}")
    for eid, name in matches[:limit]:
        print(f"  id={eid:<6} name={name!r}")
    if len(matches) > limit:
        print(f"... {len(matches) - limit} more matches not shown")

### 8.2 Helper: One-Shot Inspection by Name

In [ ]:
def inspect_entity_by_name(substr: str, top_k_neighbors: int = 10, walk_length: int = 6, num_walks: int = 2):
    """
    Convenience helper:
    - Finds entities whose name contains `substr` (case-insensitive),
    - Picks the first match,
    - Prints metadata, neighbors (semantic + cooc), and a few random walks.
    
    Args:
        substr: Search substring to find entity
        top_k_neighbors: Number of top neighbors to show
        walk_length: Length of random walks
        num_walks: Number of random walks to perform
    """
    s = substr.lower()
    matches = [
        (eid, meta["name"])
        for eid, meta in entity_meta.items()
        if s in meta["name"].lower()
    ]
    
    if not matches:
        print(f"No entities found containing {substr!r}")
        return
    
    eid, name = matches[0]
    if eid >= num_nodes:
        print(f"Matched id={eid}, name={name!r}, but id is out of range for graph (num_nodes={num_nodes}).")
        return
    
    print(f"{'=' * 70}")
    print(f"INSPECTING FIRST MATCH FOR {substr!r}")
    print(f"{'=' * 70}")
    print(f"Matched entity: id={eid} name={name!r}")
    print()
    
    # 1) Metadata + context summary
    summarize_entity(eid)
    print()
    
    # 2) Neighbors
    inspect_neighbors(eid, top_k=top_k_neighbors)
    print()
    
    # 3) Random walks
    run_random_walk(eid, walk_length=walk_length, num_walks=num_walks)

### 8.3 Helper: Compare Semantic Neighborhoods by Name

In [ ]:
def _first_match_id(substr: str):
    """
    Internal helper: return the first entity id whose name contains substr (case-insensitive),
    or None if not found.
    
    Args:
        substr: Search substring
        
    Returns:
        First matching entity ID or None
    """
    s = substr.lower()
    for eid, meta in entity_meta.items():
        if s in meta["name"].lower():
            return eid
    return None


def compare_entities_by_name(name1: str, name2: str, top_k_neighbors: int = 20):
    """
    Compare two entities (found by substring name) in terms of their semantic neighborhoods.
    
    Args:
        name1: First entity name substring
        name2: Second entity name substring
        top_k_neighbors: Number of top neighbors to compare
    """
    eid1 = _first_match_id(name1)
    eid2 = _first_match_id(name2)
    
    if eid1 is None:
        print(f"No entity found for {name1!r}")
        return
    if eid2 is None:
        print(f"No entity found for {name2!r}")
        return
    
    if eid1 >= num_nodes or eid2 >= num_nodes:
        print(f"One of the matched IDs is out of range: eid1={eid1}, eid2={eid2}, num_nodes={num_nodes}")
        return
    
    print(f"{'=' * 70}")
    print(f"COMPARING SEMANTIC NEIGHBORHOODS")
    print(f"{'=' * 70}")
    print(f"{name1!r} → id={eid1}, name={entity_meta[eid1]['name']!r}")
    print(f"{name2!r} → id={eid2}, name={entity_meta[eid2]['name']!r}")
    print()
    
    sem_ei = graph["entity", "sem", "entity"].edge_index
    sem_weight = graph["entity", "sem", "entity"].edge_weight
    
    # Get neighbors for entity 1
    mask1 = (sem_ei[0] == eid1)
    nbrs1 = sem_ei[1, mask1]
    w1 = sem_weight[mask1]
    
    # Get neighbors for entity 2
    mask2 = (sem_ei[0] == eid2)
    nbrs2 = sem_ei[1, mask2]
    w2 = sem_weight[mask2]
    
    # Sort by weight descending and get top-k
    if nbrs1.numel() > 0:
        vals1, idx1 = torch.sort(w1, descending=True)
        top_idx1 = idx1[:top_k_neighbors]
        top_nbrs1 = nbrs1[top_idx1]
        top_weights1 = vals1[:len(top_nbrs1)]
    else:
        top_nbrs1 = torch.tensor([])
        top_weights1 = torch.tensor([])
    
    if nbrs2.numel() > 0:
        vals2, idx2 = torch.sort(w2, descending=True)
        top_idx2 = idx2[:top_k_neighbors]
        top_nbrs2 = nbrs2[top_idx2]
        top_weights2 = vals2[:len(top_nbrs2)]
    else:
        top_nbrs2 = torch.tensor([])
        top_weights2 = torch.tensor([])
    
    set1 = set(top_nbrs1.tolist())
    set2 = set(top_nbrs2.tolist())
    overlap = set1 & set2
    
    print(f"Top {top_k_neighbors} semantic neighbors of {name1!r} (id={eid1}):")
    if len(top_nbrs1) > 0:
        for nid, weight in zip(top_nbrs1.tolist(), top_weights1.tolist()):
            print(f"  id={nid:<6} w={weight:.4f} name={entity_meta[nid]['name']!r}")
    else:
        print("  (no semantic neighbors)")
    print()
    
    print(f"Top {top_k_neighbors} semantic neighbors of {name2!r} (id={eid2}):")
    if len(top_nbrs2) > 0:
        for nid, weight in zip(top_nbrs2.tolist(), top_weights2.tolist()):
            print(f"  id={nid:<6} w={weight:.4f} name={entity_meta[nid]['name']!r}")
    else:
        print("  (no semantic neighbors)")
    print()
    
    print(f"Overlap in top-{top_k_neighbors} neighbors (by id): {len(overlap)}")
    if overlap:
        print("  Shared neighbors:")
        for nid in sorted(overlap):
            print(f"    id={nid:<6} name={entity_meta[nid]['name']!r}")
    else:
        print("  (no shared neighbors in top-k)")

### 8.4 Example Usage

Run the cells below to see examples of text-based entity search and comparison.
Feel free to modify the search terms or comment out examples you don't need.

In [ ]:
# Example 1: Find entities by name substring
print("=" * 70)
print("EXAMPLE 1: FIND ENTITIES BY NAME")
print("=" * 70)
print()

print("Search for 'grass':")
find_entities("grass")

print("\n" + "-" * 70 + "\n")

print("Search for 'dog':")
find_entities("dog")

In [ ]:
# Example 2: One-shot inspection by name
print("\n" + "=" * 70)
print("EXAMPLE 2: ONE-SHOT INSPECTION BY NAME")
print("=" * 70)
print()

print("Inspecting 'grass':")
inspect_entity_by_name("grass", top_k_neighbors=10, walk_length=6, num_walks=2)

In [ ]:
# Example 3: Another one-shot inspection
print("\n" + "=" * 70)
print("EXAMPLE 3: INSPECT ANOTHER ENTITY")
print("=" * 70)
print()

print("Inspecting 'dog':")
inspect_entity_by_name("dog", top_k_neighbors=10, walk_length=6, num_walks=2)

In [ ]:
# Example 4: Compare semantic neighborhoods
print("\n" + "=" * 70)
print("EXAMPLE 4: COMPARE SEMANTIC NEIGHBORHOODS")
print("=" * 70)
print()

print("Compare 'man' vs 'woman':")
compare_entities_by_name("man", "woman", top_k_neighbors=15)

In [ ]:
# Example 5: Another comparison
print("\n" + "=" * 70)
print("EXAMPLE 5: COMPARE ANOTHER PAIR")
print("=" * 70)
print()

print("Compare 'dog' vs 'cat':")
compare_entities_by_name("dog", "cat", top_k_neighbors=15)

## 9. Optional: Visualization Extras

### 9.1 PCA Projection of Entity Embeddings

In [ ]:
try:
    from sklearn.decomposition import PCA
    import numpy as np
    
    print("\n" + "=" * 70)
    print("PCA PROJECTION OF ENTITY EMBEDDINGS")
    print("=" * 70)
    
    # Sample entities for visualization
    sample_size = min(1000, num_nodes)
    idx = torch.randperm(num_nodes)[:sample_size]
    X = graph["entity"].x[idx].cpu().numpy()
    
    print(f"\nSampling {sample_size} entities for PCA...")
    print(f"  Input shape: {X.shape}")
    
    # Apply PCA
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(X)
    
    print(f"  PCA explained variance: {pca.explained_variance_ratio_}")
    print(f"  Output shape: {X2.shape}")
    
    # Get degrees for coloring
    sampled_degs = sem_deg_undirected[idx].cpu().numpy()
    
    # Plot
    plt.figure(figsize=(12, 10))
    
    # Color by semantic degree
    scatter = plt.scatter(
        X2[:, 0], X2[:, 1],
        c=sampled_degs,
        cmap='viridis',
        s=10,
        alpha=0.6
    )
    plt.colorbar(scatter, label='Semantic Degree (Undirected)')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(f'PCA Projection of {sample_size} Sampled Entity Embeddings\n(colored by semantic degree)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n  ✓ PCA visualization complete")
    
except ImportError:
    print("\n⚠ sklearn not available; skipping PCA visualization")

### 9.2 Small Subgraph Visualization (NetworkX)

In [ ]:
def visualize_subgraph(center_eid: int, radius: int = 2, max_nodes: int = 50):
    """
    Visualize a small subgraph around a center entity.
    
    Args:
        center_eid: Center entity ID
        radius: Hop distance to include
        max_nodes: Maximum nodes to include
    """
    if center_eid not in entity_meta:
        print(f"  ✗ Entity {center_eid} not found")
        return
    
    print(f"\nVisualizing subgraph around entity {center_eid}: {entity_meta[center_eid]['name']!r}")
    print(f"  Radius: {radius} hops, Max nodes: {max_nodes}")
    
    # Get ego graph (semantic edges only)
    ego = nx.ego_graph(G_sem, center_eid, radius=radius)
    
    # Limit size
    if len(ego.nodes()) > max_nodes:
        print(f"  ⚠ Subgraph too large ({len(ego.nodes())} nodes), sampling {max_nodes} nodes...")
        nodes = [center_eid] + list(ego.nodes() - {center_eid})[:max_nodes-1]
        ego = G_sem.subgraph(nodes)
    
    print(f"  Subgraph: {len(ego.nodes())} nodes, {len(ego.edges())} edges")
    
    # Create labels (show entity IDs and names)
    labels = {
        nid: f"{nid}\n{entity_meta[nid]['name'][:15]}"
        for nid in ego.nodes()
    }
    
    # Layout
    pos = nx.spring_layout(ego, k=0.5, iterations=50, seed=42)
    
    # Draw
    plt.figure(figsize=(14, 10))
    
    # Draw nodes
    node_colors = ['red' if nid == center_eid else 'lightblue' for nid in ego.nodes()]
    node_sizes = [500 if nid == center_eid else 200 for nid in ego.nodes()]
    
    nx.draw_networkx_nodes(
        ego, pos,
        node_color=node_colors,
        node_size=node_sizes,
        alpha=0.8
    )
    
    # Draw edges
    nx.draw_networkx_edges(ego, pos, alpha=0.3, width=0.5)
    
    # Draw labels
    nx.draw_networkx_labels(ego, pos, labels, font_size=7)
    
    plt.title(f"Subgraph around Entity {center_eid}: {entity_meta[center_eid]['name']!r}\n({len(ego.nodes())} nodes, {len(ego.edges())} edges)")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("  ✓ Subgraph visualization complete")

In [ ]:
# Visualize small subgraphs for sample entities
sample_ids = [0, 42, 100]
sample_ids = [eid for eid in sample_ids if eid < num_nodes]

for eid in sample_ids[:2]:  # Limit to 2 to avoid too many plots
    visualize_subgraph(eid, radius=2, max_nodes=30)

## Summary

This notebook has explored the entity-centric knowledge graph built in Phase 4:

1. **Loaded** entity graph, metadata, and context from Kaggle paths
2. **Verified** ID alignment between graph nodes and metadata/context
3. **Analyzed** degree distributions for semantic and co-occurrence edges
4. **Examined** connectivity with connected components analysis
5. **Inspected** individual entities with metadata and context
6. **Explored** neighbors (semantic and co-occurrence) for selected entities
7. **Demonstrated** random walks on the semantic graph
8. **Visualized** entity embeddings with PCA projection and small subgraphs

The graph is ready for Phase 4 query enrichment and graph search!